In [ ]:
!git clone --branch work https://github.com/danilov-e/lora-activation-steering.git
%cd /content/lora-activation-steering
%pip install -e ".[test]"

In [ ]:
!python -m pytest tests/test_core.py -q

In [ ]:
import torch

from src.config import load_config
from src.model import load_model_and_tokenizer
from src.generation import format_prompt, generate_batch

config = load_config("configs/qwen25_15b.yaml")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

model, tokenizer = load_model_and_tokenizer(config["model"])

print("Модель:", type(model).__name__)
print("Токенизатор:", type(tokenizer).__name__)
print("Устройство:", model.device)
print("Точность:", model.dtype)
print("Режим обучения:", model.training)

In [ ]:
jobs = [
    {
        "id": "smoke_1",
        "question": "What is 2 to the power of 7?",
    },
    {
        "id": "smoke_2",
        "question": "Explain in two sentences why people have different preferences about architecture.",
    },
]

print(format_prompt(tokenizer, jobs[0]["question"]))

In [ ]:
settings = dict(config["generation"])
settings["max_new_tokens"] = 128

rows = generate_batch(
    model,
    tokenizer,
    jobs,
    settings,
    seed=config["run"]["seed"],
)

for row in rows:
    print("\nВопрос:", row["question"])
    print("Ответ:", row["answer"])
    print("Токенов:", row["generated_tokens"])
    print("Причина остановки:", row["finish_reason"])

if model.device.type == "cuda":
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"\nПиковая память тензоров PyTorch: {peak:.2f} GiB")